In [ ]:
!pip install -q "torchao>=0.16.0"

from diffusers import StableDiffusionPipeline
import torch
import matplotlib.pyplot as plt

print("Baixando o Stable Diffusion base...\n")
pipeline = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5", 
    torch_dtype=torch.float16
).to("cuda")

prompts = [
    "estilo_rupestre, uma pintura de um animal com chifres em uma parede de rocha",
    "estilo_rupestre, uma pintura de um grupo de pessoas caçando com lanças",
    "estilo_rupestre, o desenho de um pássaro",
    "estilo_rupestre, uma pintura de um animal com chifres e um pássaro",
    "estilo_rupestre, uma pintura de pessoas dançando",
    "estilo_rupestre, uma pintura de uma grande família de animais selvagens"
]

semente = 42

imagens_base = []
imagens_v1 = []
imagens_v2 = []

print("Gerando 6 imagens com o Modelo BASE (Sem treino)...")
for texto in prompts:
    gerador = torch.Generator("cuda").manual_seed(semente)
    img = pipeline(texto, num_inference_steps=30, generator=gerador).images[0]
    imagens_base.append(img)


print("Gerando 6 imagens com o Modelo V1...")
pipeline.load_lora_weights("LuisCurado/lora-estilo-rupestre")
for texto in prompts:
    gerador = torch.Generator("cuda").manual_seed(semente)
    img = pipeline(texto, num_inference_steps=30, generator=gerador).images[0]
    imagens_v1.append(img)
pipeline.unload_lora_weights() 


print("Gerando 6 imagens com o Modelo V2...")
pipeline.load_lora_weights("LuisCurado/lora-estilo-rupestre-v2")
for texto in prompts:
    gerador = torch.Generator("cuda").manual_seed(semente)
    img = pipeline(texto, num_inference_steps=30, generator=gerador).images[0]
    imagens_v2.append(img)
pipeline.unload_lora_weights() 


print("\nConcluído! Montando a grade 6x3...")
fig, axs = plt.subplots(6, 3, figsize=(15, 24))

for i in range(6):
    # Coluna 1: Modelo Base
    axs[i, 0].imshow(imagens_base[i])
    axs[i, 0].set_title(f"Base - Prompt {i+1}")
    axs[i, 0].axis('off')
    
    # Coluna 2: Modelo V1
    axs[i, 1].imshow(imagens_v1[i])
    axs[i, 1].set_title(f"V1 - Prompt {i+1}")
    axs[i, 1].axis('off')
    
    # Coluna 3: Modelo V2
    axs[i, 2].imshow(imagens_v2[i])
    axs[i, 2].set_title(f"V2 - Prompt {i+1}")
    axs[i, 2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Instala as bibliotecas de métricas
!pip -q install torchmetrics torchvision

import torch
import torchvision.transforms as T
from torchmetrics.multimodal.clip_score import CLIPScore

print("Baixando o modelo avaliador CLIP...\n")
# Instancia a métrica conforme exigido no roteiro
metrica = CLIPScore(model_name_or_path="openai/clip-vit-base-patch16").to("cuda")

# Função que transforma a imagem no 'tensor_da_imagem' e calcula a média
def calcular_media_clip(lista_de_imagens, lista_de_textos):
    transformar_em_tensor = T.ToTensor()
    soma = 0
    
    for img, texto in zip(lista_de_imagens, lista_de_textos):
        # Transforma a imagem PIL em um Tensor matemático e envia para a Placa de Vídeo
        tensor_da_imagem = transformar_em_tensor(img).to("cuda")
        
        # Calcula o score individual
        score = metrica(tensor_da_imagem, texto)
        soma += score.item()
        
    # Retorna a média do modelo
    return soma / len(lista_de_imagens)

print("Calculando as pontuações (isso leva alguns segundos)...\n")

# Calcula as médias usando as listas que já estão na memória do Colab
media_base = calcular_media_clip(imagens_base, prompts)
media_v1 = calcular_media_clip(imagens_v1, prompts)
media_v2 = calcular_media_clip(imagens_v2, prompts)

print(f"📊 CLIPScore Médio do Modelo BASE: {media_base:.2f}")
print(f"📊 CLIPScore Médio do Modelo V1:   {media_v1:.2f}")
print(f"📊 CLIPScore Médio do Modelo V2:   {media_v2:.2f}")